# 04 — Model Training

Train a pair classifier using a Source-1-level split to reduce leakage.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
ROOT=Path.cwd().resolve()
if not (ROOT/'data').exists() and (ROOT.parent/'data').exists(): ROOT=ROOT.parent
features=pd.read_pickle(ROOT/'output'/'training_features.pkl')
feature_cols=['name_similarity','name_token_similarity','address_similarity','address_token_similarity','name_exact','country_match','name_missing','address_missing','name_length_diff','address_length_diff']
X=features[feature_cols]; y=features['label']; groups=features['source1_entity_id']
gss=GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=42)
tr,va=next(gss.split(X,y,groups=groups))
model=RandomForestClassifier(n_estimators=250,random_state=42,n_jobs=-1,class_weight='balanced_subsample',min_samples_leaf=2,max_features='sqrt')
model.fit(X.iloc[tr],y.iloc[tr])
print('train pairs:',len(tr),'validation pairs:',len(va),'positive rate:',y.mean())

In [ ]:
joblib.dump(model,ROOT/'models'/'entity_resolution_random_forest.joblib')
import json
(ROOT/'models'/'feature_columns.json').write_text(json.dumps(feature_cols,indent=2))
print('Model saved.')

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
p=model.predict_proba(X.iloc[va])[:,1]
for t in [0.30,0.40,0.50,0.60,0.70,0.80,0.90]:
    pred=(p>=t).astype(int)
    _,_,f,_=precision_recall_fscore_support(y.iloc[va],pred,beta=0.5,average='binary',zero_division=0)
    print(t, 'pair F0.5=',round(f,5))